# Loan Data Analysis

## Task 1: Create a Normalized Database (3NF)

In [ ]:
import sqlite3
import csv
import pandas as pd

# Create database connection
conn = sqlite3.connect('loan_database.db')
cursor = conn.cursor()

# Create tables in 3NF

# 1. Education Level (Lookup table)
cursor.execute('''
CREATE TABLE IF NOT EXISTS education_level (
    education_id INTEGER PRIMARY KEY,
    education_name TEXT UNIQUE
)''')

# 2. Home Ownership (Lookup table)
cursor.execute('''
CREATE TABLE IF NOT EXISTS home_ownership (
    ownership_id INTEGER PRIMARY KEY,
    ownership_type TEXT UNIQUE
)''')

# 3. Loan Intent (Lookup table)
cursor.execute('''
CREATE TABLE IF NOT EXISTS loan_intent (
    intent_id INTEGER PRIMARY KEY,
    intent_name TEXT UNIQUE
)''')

# 4. Person table (Contains personal information)
cursor.execute('''
CREATE TABLE IF NOT EXISTS person (
    person_id INTEGER PRIMARY KEY AUTOINCREMENT,
    age INTEGER,
    gender TEXT,
    education_level_id INTEGER,
    income FLOAT,
    employment_experience INTEGER,
    home_ownership_id INTEGER,
    credit_history_length INTEGER,
    credit_score INTEGER,
    has_previous_defaults BOOLEAN,
    FOREIGN KEY (education_level_id) REFERENCES education_level(education_id),
    FOREIGN KEY (home_ownership_id) REFERENCES home_ownership(ownership_id)
)''')

# 5. Loan table (Contains loan information)
cursor.execute('''
CREATE TABLE IF NOT EXISTS loan (
    loan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    person_id INTEGER,
    loan_amount FLOAT,
    intent_id INTEGER,
    interest_rate FLOAT,
    percent_income FLOAT,
    loan_status BOOLEAN,
    FOREIGN KEY (person_id) REFERENCES person(person_id),
    FOREIGN KEY (intent_id) REFERENCES loan_intent(intent_id)
)''')

# Load data from CSV
with open('/Users/dhinakaryalla/Downloads/loan_data.csv', 'r') as file:
    csv_reader = csv.DictReader(file)
    
    # Collect unique values for lookup tables
    education_levels = set()
    home_ownership_types = set()
    loan_intents = set()
    
    data = []
    for row in csv_reader:
        education_levels.add(row['person_education'])
        home_ownership_types.add(row['person_home_ownership'])
        loan_intents.add(row['loan_intent'])
        data.append(row)

    # Populate lookup tables
    for edu in education_levels:
        cursor.execute('INSERT OR IGNORE INTO education_level (education_name) VALUES (?)', (edu,))
    
    for ownership in home_ownership_types:
        cursor.execute('INSERT OR IGNORE INTO home_ownership (ownership_type) VALUES (?)', (ownership,))
    
    for intent in loan_intents:
        cursor.execute('INSERT OR IGNORE INTO loan_intent (intent_name) VALUES (?)', (intent,))

    # Function to get ID from lookup tables
    def get_lookup_id(table, column, value):
        cursor.execute(f'SELECT {table[:-6]}_id FROM {table} WHERE {column} = ?', (value,))
        return cursor.fetchone()[0]

    # Insert main data
    for row in data:
        # Insert person
        cursor.execute('''
        INSERT INTO person (
            age, gender, education_level_id, income, employment_experience,
            home_ownership_id, credit_history_length, credit_score, has_previous_defaults
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            int(float(row['person_age'])),
            row['person_gender'],
            get_lookup_id('education_level', 'education_name', row['person_education']),
            float(row['person_income']),
            int(float(row['person_emp_exp'])),
            get_lookup_id('home_ownership', 'ownership_type', row['person_home_ownership']),
            int(float(row['cb_person_cred_hist_length'])),
            int(float(row['credit_score'])),
            row['previous_loan_defaults_on_file'] == 'Yes'
        ))
        
        person_id = cursor.lastrowid
        
        # Insert loan
        cursor.execute('''
        INSERT INTO loan (
            person_id, loan_amount, intent_id, interest_rate, percent_income, loan_status
        ) VALUES (?, ?, ?, ?, ?, ?)
        ''', (
            person_id,
            float(row['loan_amnt']),
            get_lookup_id('loan_intent', 'intent_name', row['loan_intent']),
            float(row['loan_int_rate']),
            float(row['loan_percent_income']),
            int(row['loan_status'])
        ))

conn.commit()
print("Database created and normalized to 3NF successfully!")

## Task 2: Write SQL Join Statement to Fetch Data into Pandas DataFrame

In [ ]:
# Complex SQL query joining all tables with additional calculated fields
query = '''
WITH loan_metrics AS (
    SELECT 
        intent_id,
        AVG(interest_rate) as avg_intent_interest_rate
    FROM loan
    GROUP BY intent_id
)
SELECT 
    -- Person Information
    p.person_id,
    p.age,
    p.gender,
    el.education_name as education_level,
    p.income,
    p.employment_experience,
    ho.ownership_type as home_ownership,
    p.credit_history_length,
    p.credit_score,
    p.has_previous_defaults,
    
    -- Loan Information
    l.loan_id,
    l.loan_amount,
    li.intent_name as loan_purpose,
    l.interest_rate,
    l.percent_income,
    l.loan_status,
    
    -- Calculated Fields
    CASE 
        WHEN p.credit_score >= 700 THEN 'Excellent'
        WHEN p.credit_score >= 650 THEN 'Good'
        WHEN p.credit_score >= 600 THEN 'Fair'
        ELSE 'Poor'
    END as credit_category,
    
    CASE 
        WHEN l.loan_status = 1 THEN 'Approved'
        ELSE 'Denied'
    END as loan_status_desc,
    
    -- Risk Metrics
    ROUND(l.interest_rate - lm.avg_intent_interest_rate, 2) as interest_rate_variance,
    ROUND(l.loan_amount / NULLIF(p.income, 0) * 100, 2) as debt_to_income_ratio,
    
    -- Demographic Grouping
    CASE 
        WHEN p.age < 25 THEN 'Young'
        WHEN p.age < 35 THEN 'Adult'
        ELSE 'Senior'
    END as age_category,
    
    -- Experience Level
    CASE 
        WHEN p.employment_experience < 2 THEN 'Entry'
        WHEN p.employment_experience < 5 THEN 'Mid-level'
        ELSE 'Senior'
    END as experience_level
    
FROM person p
-- Join with loan table
JOIN loan l 
    ON p.person_id = l.person_id

-- Join with education_level lookup table
JOIN education_level el 
    ON p.education_level_id = el.education_id

-- Join with home_ownership lookup table
JOIN home_ownership ho 
    ON p.home_ownership_id = ho.ownership_id

-- Join with loan_intent lookup table
JOIN loan_intent li 
    ON l.intent_id = li.intent_id

-- Join with loan metrics CTE
JOIN loan_metrics lm
    ON l.intent_id = lm.intent_id

-- Add some basic filtering
WHERE p.income > 0
  AND p.credit_score > 0

-- Order by person_id
ORDER BY p.person_id
'''

# Execute query and load into DataFrame
df = pd.read_sql_query(query, conn)

# Display sample of the data
print("Data loaded successfully! Sample of the joined data:")
display(df.head())

# Display summary statistics
print("\nSummary Statistics:")
summary_stats = df.agg({
    'loan_amount': ['count', 'mean', 'min', 'max'],
    'interest_rate': ['mean', 'min', 'max'],
    'credit_score': ['mean', 'min', 'max'],
    'income': ['mean', 'min', 'max']
}).round(2)
display(summary_stats)

# Close the connection
conn.close()